In [19]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import calendar 
import re
from typing import List

new_data_dir = 'new_data'
sns.set_theme(style="whitegrid")

# --- (Keep your load_and_prepare_pendle_file function as you provided it) ---
def load_and_prepare_pendle_file(filepath: str, asset_name_from_parser: str, chain_from_parser: str, 
                                 tenor_month_from_parser: str, tenor_year_from_parser: str,
                                 expected_apy_cols: List[str] | None = None) -> pd.DataFrame | None:
    """
    Loads and preprocesses a single Pendle data CSV.
    Converts APYs to decimal if they appear to be percentages.
    Resamples to daily frequency, taking the last observation of the day.
    """
    if expected_apy_cols is None:
        expected_apy_cols = ['impliedApy', 'underlyingApy1DMA', 'underlyingApy7DMA']

    try:
        df = pd.read_csv(filepath)
        if 'time' not in df.columns:
            print(f"  Warning: 'time' column not found in {filepath}. Skipping.")
            return None
        
        df['time_cleaned'] = df['time'].astype(str).str.replace(r'GMT.*$', '', regex=True).str.strip()
        df['datetime'] = pd.to_datetime(df['time_cleaned'], errors='coerce')
        df.dropna(subset=['datetime'], inplace=True)
        if df.empty:
            print(f"  Warning: No valid datetime entries in {filepath} after cleaning. Skipping.")
            return None
        df = df.set_index('datetime')
        
        rename_map = {}
        final_cols_map = {} # Maps generic key to actual column name in df after rename
        if 'impliedApy' in df.columns and 'impliedApy' in expected_apy_cols:
            rename_map['impliedApy'] = 'implied_apy'
            final_cols_map['implied_apy'] = 'implied_apy'
        if 'underlyingApy1DMA' in df.columns and 'underlyingApy1DMA' in expected_apy_cols:
            rename_map['underlyingApy1DMA'] = 'underlying_apy_1dma'
            final_cols_map['underlying_apy_1dma'] = 'underlying_apy_1dma'
        if 'underlyingApy7DMA' in df.columns and 'underlyingApy7DMA' in expected_apy_cols:
            rename_map['underlyingApy7DMA'] = 'underlying_apy_7dma'
            final_cols_map['underlying_apy_7dma'] = 'underlying_apy_7dma'
        
        df.rename(columns=rename_map, inplace=True)

        for generic_key, actual_col_name in final_cols_map.items():
            if actual_col_name in df.columns:
                df[actual_col_name] = pd.to_numeric(df[actual_col_name], errors='coerce')
                # Ensure APYs are decimal (e.g., 0.05 for 5%)
                mean_val = df[actual_col_name].abs().mean()
                if not pd.isna(mean_val) and mean_val > 1.0 and mean_val != 0:
                    print(f"  Converting {actual_col_name} in {filepath} from % to decimal.")
                    df[actual_col_name] /= 100.0
            else:
                print(f"  Warning: Expected APY column '{generic_key}' (mapped to '{actual_col_name}') not found in {filepath}")
                
        daily_df = df.resample('D').last()
        daily_df.dropna(how='all', inplace=True)

        if daily_df.empty:
            print(f"  Warning: No data after resampling for {filepath}")
            return None

        daily_df['protocol_asset_raw'] = asset_name_from_parser 
        daily_df['chain'] = chain_from_parser
        daily_df['tenor_month_parsed'] = tenor_month_from_parser # Store parsed month
        daily_df['tenor_year_parsed'] = tenor_year_from_parser   # Store parsed year
        daily_df['tenor_str'] = f"{tenor_month_from_parser}{tenor_year_from_parser}"
        
        month_map = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6, 
                     'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
        try:
            year_full = int(f"20{tenor_year_from_parser}")
            month_num = month_map[tenor_month_from_parser]
            last_day = calendar.monthrange(year_full, month_num)[1]
            expiry_date_str = f"{year_full}-{month_num:02d}-{last_day:02d}"
            expiry_date = pd.to_datetime(expiry_date_str)
            daily_df['TTM_days'] = (expiry_date - daily_df.index).days
            daily_df['TTM_years'] = np.maximum(0, daily_df['TTM_days'] / 365.25)
        except (ValueError, KeyError) as e_ttm:
             print(f"  Could not parse expiry for TTM for {filepath} ({tenor_month_from_parser}{tenor_year_from_parser}): {e_ttm}. TTM will be NaN.")
             daily_df['TTM_years'] = np.nan

        cols_to_keep = ['implied_apy', 'underlying_apy_1dma', 'underlying_apy_7dma', 'volume', 
                        'protocol_asset_raw', 'chain', 'tenor_str', 'TTM_years']
        final_present_cols = [col for col in cols_to_keep if col in daily_df.columns]
        
        return daily_df[final_present_cols]
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return None
    except Exception as e:
        print(f"Error processing file {filepath}: {e}")
        return None

def parse_pendle_filename(filename_without_ext: str) -> tuple | None:
    """
    Parses a Pendle filename to extract chain, tenor month, tenor year, and asset name.
    Expected format: CHAIN_MONTHYEAR_ASSET.csv (e.g., ETH_Jun25_weETHs.csv)
                     or CHAIN_MONTHYEAR_ASSET_VARIANT.csv (e.g. ETH_Jun24_weETH_z.csv - less common)
    """
    known_base_assets_with_variants = ["weETH", "rsETH"] # Add others if needed
    
    parts = filename_without_ext.split('_')
    if len(parts) < 3:
        print(f"Warning: Filename '{filename_without_ext}' does not match expected format CHAIN_TENOR_ASSET. Skipping.")
        return None

    chain = parts[0]
    tenor_full = parts[1] # e.g., Jun25, Apr24, Dec24, Oct25
    
    # Extract asset name (can be multiple parts if asset name has underscores)
    asset_parts = parts[2:]
    asset_name = "_".join(asset_parts)

    # Extract month and year from tenor_full
    # Use regex to find the first sequence of letters (month) and then digits (year)
    match = re.match(r"([A-Za-z]+)([0-9]+)", tenor_full)
    if not match:
        print(f"Warning: Could not parse tenor '{tenor_full}' from filename '{filename_without_ext}'. Skipping.")
        return None
        
    tenor_month_str = match.group(1).capitalize() # E.g., Jun, Apr, Dec
    tenor_year_str = match.group(2) # E.g., 25, 24

    # Validate month
    valid_months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    if tenor_month_str not in valid_months:
        print(f"Warning: Invalid month '{tenor_month_str}' in filename '{filename_without_ext}'. Skipping.")
        return None
        
    return chain, tenor_month_str, tenor_year_str, asset_name

# --- Main Data Loading Loop ---
all_pendle_data_nested = {} # Structure: {'Chain': {'AssetRaw': {'TenorStr': DataFrame}}}
dfs_to_concat = []

if not os.path.isdir(new_data_dir):
    print(f"Directory not found: {new_data_dir}")
else:
    for filename in os.listdir(new_data_dir):
        if filename.endswith(".csv"):
            filepath = os.path.join(new_data_dir, filename)
            filename_no_ext = filename[:-4]
            
            parsed_info = parse_pendle_filename(filename_no_ext)
            if parsed_info:
                chain, tenor_month, tenor_year, asset_name = parsed_info
                
                print(f"Processing: {filename} -> Chain: {chain}, Tenor: {tenor_month}{tenor_year}, Asset: {asset_name}")
                
                df = load_and_prepare_pendle_file(filepath, asset_name, chain, tenor_month, tenor_year)
                
                if df is not None and not df.empty:
                    # Populate the nested dictionary
                    if chain not in all_pendle_data_nested:
                        all_pendle_data_nested[chain] = {}
                    if asset_name not in all_pendle_data_nested[chain]:
                        all_pendle_data_nested[chain][asset_name] = {}
                    
                    tenor_string = f"{tenor_month}{tenor_year}" # e.g. Jun24, Dec25
                    all_pendle_data_nested[chain][asset_name][tenor_string] = df
                    dfs_to_concat.append(df)
                    print(f"  Successfully loaded and stored: {chain} - {asset_name} - {tenor_string}")
                else:
                    print(f"  Skipped storing data for {filename} due to loading/processing issues.")
    if dfs_to_concat:
        master_df = pd.concat(dfs_to_concat, ignore_index=False) 
        master_df.sort_index(inplace=True)
    else :
        master_df = pd.DataFrame()

print("\n--- Summary of Loaded Data Structure ---")
for chain_key, assets_dict in all_pendle_data_nested.items():
    print(f"Chain: {chain_key}")
    for asset_key, tenors_dict in assets_dict.items():
        print(f"  Asset: {asset_key}, Tenors: {list(tenors_dict.keys())}")

Processing: ARB_Apr24_rsETH.csv -> Chain: ARB, Tenor: Apr24, Asset: rsETH
  Successfully loaded and stored: ARB - rsETH - Apr24
Processing: ARB_Apr24_weETH.csv -> Chain: ARB, Tenor: Apr24, Asset: weETH
  Successfully loaded and stored: ARB - weETH - Apr24
Processing: ARB_Dec24_rsETH.csv -> Chain: ARB, Tenor: Dec24, Asset: rsETH
  Successfully loaded and stored: ARB - rsETH - Dec24
Processing: ARB_Dec24_uniETH.csv -> Chain: ARB, Tenor: Dec24, Asset: uniETH
  Successfully loaded and stored: ARB - uniETH - Dec24
Processing: ARB_Dec24_weETH.csv -> Chain: ARB, Tenor: Dec24, Asset: weETH
  Successfully loaded and stored: ARB - weETH - Dec24
Processing: ARB_Jun24_ezETH.csv -> Chain: ARB, Tenor: Jun24, Asset: ezETH
  Successfully loaded and stored: ARB - ezETH - Jun24
Processing: ARB_Jun24_rsETH.csv -> Chain: ARB, Tenor: Jun24, Asset: rsETH
  Successfully loaded and stored: ARB - rsETH - Jun24
Processing: ARB_Jun24_weETH.csv -> Chain: ARB, Tenor: Jun24, Asset: weETH
  Successfully loaded and s

## Processing master_df

In [ ]:
def preprocess_master_df(master_df: pd.DataFrame) -> pd.DataFrame:
    # styETH benchmark APY data
    try: 
        styeth_df = pd.read_csv('new_data/benchmark/ETH_styETH.csv')
        styeth_df['datetime'] = pd.to_datetime(styeth_df['time'])
        styeth_df.set_index('datetime', inplace=True)
        styeth_df = styeth_df[['styeth_apy']].rename(columns={'styeth_apy': 'STYETH'})
        if styeth_df['STYETH'].abs().mean() > 1.0:
            styeth_df['STYETH'] /= 100.0
    except FileNotFoundError:
        print("Error: File not found for ETH styETH data.")
        master_df['STYETH'] = np.nan

    # enrich with styETH APY
    styeth_df.index = styeth_df.index.normalize()
    master_df = master_df.merge(styeth_df, on='datetime', how='left')

    # tvl data 

    return master_df


master_df = preprocess_master_df(master_df)
master_df

              STYETH
datetime            
2025-06-26  0.030201
2025-06-25  0.030345
2025-06-24  0.031397
2025-06-23  0.030728
2025-06-22  0.029868


,implied_apy,underlying_apy_1dma,underlying_apy_7dma,volume,protocol_asset_raw,chain,tenor_str,TTM_years,STYETH_x,STYETH_y
datetime,,,,,,,,,,
2023-03-24,0.0578,0.0470,0.0544,0.0,wstETH,ETH,Dec24,1.774127,0.045557,0.045557
2023-03-25,0.0578,0.0427,0.0126,0.0,wstETH,ETH,Dec24,1.771389,0.044390,0.044390
2023-03-26,0.0578,0.0422,0.0186,0.0,wstETH,ETH,Dec24,1.768652,0.044522,0.044522
2023-03-27,0.0578,0.0465,0.0252,0.0,wstETH,ETH,Dec24,1.765914,0.046069,0.046069
2023-03-28,0.0578,0.0542,0.0463,0.0,wstETH,ETH,Dec24,1.763176,0.046159,0.046159
...,...,...,...,...,...,...,...,...,...,...
2025-06-23,0.0342,0.0246,0.0267,0.0,rsETH,ARB,Jun25,0.019165,0.030728,0.030728
2025-06-23,0.0951,0.0127,0.0457,0.0,uniETH,ETH,Jun25,0.019165,0.030728,0.030728
2025-06-23,0.0272,0.0303,0.0282,0.0,wstETH,ETH,Dec25,0.522930,0.030728,0.030728
